In [6]:
import open3d as o3d
import numpy as np
import os
import sys

### prepare input data
a point cloud is loaded and downsampled using `voxel_downs_ample()`

In [7]:
print("load a ply point cloud, print it, and render it")
pcd = o3d.io.read_point_cloud(r"C:\Users\Jin Cai\Desktop\open3D_tutorial\data\extract\DemoCropPointCloud\fragment.ply")
o3d.visualization.draw_geometries([pcd])

print("downsample the point cloud with a voxel of 0.02")
voxel_down_pcd = pcd.voxel_down_sample(voxel_size = 0.02)
o3d.visualization.draw_geometries([voxel_down_pcd])

load a ply point cloud, print it, and render it
downsample the point cloud with a voxel of 0.02


### Select down sample 选择性下采样
The following helper function uses `select_by_index`, which takes a binary mask to output only the selected points. The selected points and the non-selected points are visualized.

以下辅助函数使用 `select_by_index`，该函数通过二值掩码（binary mask）来筛选并输出选中的点。  
随后，将选中的点和未选中的点分别进行可视化。

In [8]:
#用def定义一个名为display_inlier_outlier的函数，参数包括cloud（输入的原始点云）和ind（inlier的index）
def display_inlier_outlier(cloud, ind):
    inlier_cloud = cloud.select_by_index(ind) #提取内点 
    outlier_cloud = cloud.select_by_index(ind, invert=True) #提取离群点 

    print("Showing outliers (red) and inliers (gray): ")
    #paint_unidfor_color用于给整个点云设置统一的颜色
    outlier_cloud.paint_uniform_color([1, 0, 0])
    inlier_cloud.paint_uniform_color([0.8, 0.8, 0.8])
    o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud])

### Statistical outlier removal 统计离群点去除 SOR
`statistical_outlier_removal` removes points that are further away from their neighbors compared to the average for the point cloud. It takes two input parameters:
- `nb_neighbors`, which specifies how many neighbors are taken into account in order to calculate the average distance for a given point.
- `std_ratio`, which allows setting the threshold level based on the standard deviation of the average distances across the point cloud. The lower this number the more aggressive the filter will be.

`statistical_outlier_removal` 用于去除那些与邻近点之间的距离相较于整个点云的平均水平明显偏大的点。
该函数需要两个输入参数：
- `nb_neighbors`（邻近点数量）：指定在计算每个点的平均邻近距离时考虑的邻近点数量。决定局部邻域的大小
- `std_ratio`（标准差倍数）：控制离群点判定阈值的标准差倍数。该数值越小，滤波越严格，即越多的点可能被判定为离群点并被去除。

In [ ]:
print("Statistical outlier removal")

#用体素下采样后的点云进行离群点去除
#cl：去除离群点后保留下来的点云，即inlier
#ind：这些保留下来的点在原点云中的index
#原始的voxel_down_pcd不会被修改
cl, ind = voxel_down_pcd.remove_statistical_outlier(
    nb_neighbors=20, #计算每个点的平均临近距离时，考虑最近的20个点
    std_ratio=2.0  #设2倍标准差来确定离群点的判定阈值。这里的数值越低，滤波越严格，得到的红色离群点也就越多
)

#调用上面定义的display_inlier_outlier函数
display_inlier_outlier(voxel_down_pcd, ind)

Statistical outlier removal
Showing outliers (red) and inliers (gray): 


### Radius outlier removal 半径离群点去除ROR
`radius_outlier_removal` removes points that have few neighbors in a given sphere around them. Two parameters can be used to tune the filter to your data:
- `nb_points`, which lets you pick the minimum amount of points that the sphere should contain.
- `radius`, which defines the radius of the sphere that will be used for counting the neighbors.

`radius_outlier_removal` 用于去除在指定球形邻域内邻近点数量过少的点。
可以通过以下两个参数，根据点云数据的特点调整滤波器：
- `nb_points`（最小邻近点数量）：用于指定球形邻域内必须包含的最少点数。 越大，滤波约严格
- `radius`（搜索半径）：用于定义球形邻域的半径，该球形区域用于统计某个点周围的邻近点数量。越小，通常越严格

ROR 的基本原理：   
对于点云中的每个点，以该点为球心，建立半径为 radius 的球形邻域。  
统计该邻域内的点数，如果数量不足`nb_points`，则将该点判定为离群点并去除。  
ROR 根据固定半径内的邻近点数量判断离群点，而 SOR 根据邻近点平均距离的统计分布判断离群

In [ ]:
print("Radius outlier removal")

cl, ind = voxel_down_pcd.remove_radius_outlier(
    nb_points=16,   #点P邻居最少有16个，若小于16个，则点P被判定为离群点
    radius=0.05  #点P的球形邻域半径为0.05
    #可以试着改一下这两个参数的值，看有啥效果
)

display_inlier_outlier(voxel_down_pcd, ind)

Radius outlier removal
Showing outliers (red) and inliers (gray): 


SOR利用点之间的平均距离，更适合去除统计意义上异常的稀疏噪声点。适合预处理和点云配准。
ROR主要依赖密度，更适合在点云密度比较均匀时，去除局部邻近点数量不足的孤立点或小型噪声簇。更适合密度较稳定的点云清理以及孤立点去除。